## Disable Flash Attention

In [49]:
import os
os.environ["DISABLE_TRANSFORMERS_FLASH_ATTN"] = "1"

## Load dataset

In [50]:
import pandas as pd
df = df = pd.read_csv("quality_data.csv")
print(df.head())

                      narration        mode    type    category   subcategory
0        emi auto debit sbi acc  AUTO_DEBIT   Debit     expense          loan
1      salary credited via hdfc        NEFT  Credit      income        salary
2             upi txn to zomato         UPI   Debit     expense          food
3  paid electricity bill online  NETBANKING   Debit     expense         bills
4      mutual fund sip deducted  AUTO_DEBIT   Debit  investment  mutual_funds


# Create input_text

In [51]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [52]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [53]:
df["label"] = df["category"] + "_" + df["subcategory"]

In [54]:
print(df[["input_text", "label"]].head())

                                          input_text                    label
0  narration: emi auto debit sbi acc | mode: auto...             expense_loan
1  narration: salary credited via hdfc | mode: ne...            income_salary
2  narration: upi txn to zomato | mode: upi | typ...             expense_food
3  narration: paid electricity bill online | mode...            expense_bills
4  narration: mutual fund sip deducted | mode: au...  investment_mutual_funds


## Prepare Dataset for HuggingFace

In [55]:
from datasets import Dataset

df["input_text"] = (
    df["narration"] + " " + df["mode"] + " " + df["type"]
)

df["input_text"] = df["input_text"].str.lower()

dataset = Dataset.from_pandas(df[["input_text", "label"]])

## Encode Labels to IDs

In [56]:
labels = list(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def encode(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

## Tokenization

In [57]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["input_text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

## Train-Test Split

In [58]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

## Load Model + Training Setup

In [59]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=6,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/240 [00:00<?, ?it/s]

{'loss': 2.63, 'grad_norm': 4.965490341186523, 'learning_rate': 4.791666666666667e-05, 'epoch': 0.25}
{'loss': 2.5026, 'grad_norm': 6.310272693634033, 'learning_rate': 4.5833333333333334e-05, 'epoch': 0.5}
{'loss': 2.5732, 'grad_norm': 6.210803508758545, 'learning_rate': 4.375e-05, 'epoch': 0.75}
{'loss': 2.1587, 'grad_norm': 8.236345291137695, 'learning_rate': 4.166666666666667e-05, 'epoch': 1.0}
{'loss': 1.9418, 'grad_norm': 6.941483974456787, 'learning_rate': 3.958333333333333e-05, 'epoch': 1.25}
{'loss': 1.8909, 'grad_norm': 8.236449241638184, 'learning_rate': 3.7500000000000003e-05, 'epoch': 1.5}
{'loss': 1.6573, 'grad_norm': 7.747232437133789, 'learning_rate': 3.541666666666667e-05, 'epoch': 1.75}
{'loss': 1.4715, 'grad_norm': 8.999961853027344, 'learning_rate': 3.3333333333333335e-05, 'epoch': 2.0}
{'loss': 1.4289, 'grad_norm': 6.6645894050598145, 'learning_rate': 3.125e-05, 'epoch': 2.25}
{'loss': 1.0066, 'grad_norm': 5.485720157623291, 'learning_rate': 2.916666666666667e-05, '

TrainOutput(global_step=240, training_loss=1.1372462650140127, metrics={'train_runtime': 2213.2189, 'train_samples_per_second': 0.428, 'train_steps_per_second': 0.108, 'total_flos': 125608207749120.0, 'train_loss': 1.1372462650140127, 'epoch': 6.0})

## Trainer Setup + Model Training

In [60]:
def explain_distilbert(text, tokenizer, model, labels, top_n=3):
    import torch

    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    probs = torch.softmax(logits, dim=1)[0].tolist()

    # Top prediction
    top_idx = probs.index(max(probs))
    top_label = labels[top_idx]
    top_prob = probs[top_idx]

    # Second best
    sorted_indices = sorted(range(len(probs)), key=lambda i: probs[i])
    second_idx = sorted_indices[-2]
    second_label = labels[second_idx]
    second_prob = probs[second_idx]

    # Tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tokens = [t.replace("##", "") for t in tokens if t not in ["[CLS]", "[SEP]", "[PAD]"]]
    tokens = [t for t in tokens if len(t) > 2][:top_n]

    return f"Tokens [{', '.join(tokens)}] → '{top_label}' ({top_prob:.2f}), next '{second_label}' ({second_prob:.2f})"

## Explainability Function

In [61]:
import torch
import torch.nn.functional as F

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    confidence = torch.max(probs).item()
    pred_id = torch.argmax(probs).item()

    label = id2label[pred_id]
    category, subcategory = label.split("_", 1)

    return category, subcategory, round(confidence, 2)

def confidence_level(conf):
    if conf >= 0.65:
        return "High"
    elif conf >= 0.4:
        return "Medium"
    else:
        return "Low"

## Prediction Function

In [62]:
results = []

for i in range(len(df)):
    narration = df["narration"].iloc[i]
    mode = df["mode"].iloc[i]
    txn_type = df["type"].iloc[i]
    text = df["input_text"].iloc[i]

    pred_category, pred_subcategory, confidence = predict(text)

    # 🔥 ADD THIS LINE
    model_comment = explain_distilbert(text, tokenizer, model, labels)

    results.append({
        "narration": narration,
        "mode": mode,
        "type": txn_type,
        "category": df["category"].iloc[i],
        "subcategory": df["subcategory"].iloc[i],
        "predicted_category": pred_category,
        "predicted_subcategory": pred_subcategory,
        "confidence": confidence,
        "confidence_percent": f"{round(confidence*100,1)}%",
        "confidence_level": confidence_level(confidence),
        "correct": (df["category"].iloc[i] == pred_category) and 
                   (df["subcategory"].iloc[i] == pred_subcategory),

        # ✅ ADD THIS
        "model_comment": model_comment
    })

## Confidence Level Classification

In [63]:
result_df = pd.DataFrame(results)

In [64]:
result_df.head(20)

,narration,mode,type,category,subcategory,predicted_category,predicted_subcategory,confidence,confidence_percent,confidence_level,correct,model_comment
0,emi auto debit sbi acc,AUTO_DEBIT,Debit,expense,loan,expense,loan,0.78,78.0%,High,True,"Tokens [emi, auto, bit] → 'expense_loan' (0.85..."
1,salary credited via hdfc,NEFT,Credit,income,salary,income,salary,0.91,91.0%,High,True,"Tokens [salary, credited, via] → 'income_salar..."
2,upi txn to zomato,UPI,Debit,expense,food,expense,food,0.75,75.0%,High,True,"Tokens [oma, bit] → 'expense_food' (0.76), nex..."
3,paid electricity bill online,NETBANKING,Debit,expense,bills,expense,bills,0.89,89.0%,High,True,"Tokens [paid, electricity, bill] → 'expense_bi..."
4,mutual fund sip deducted,AUTO_DEBIT,Debit,investment,mutual_funds,expense,loan,0.23,23.0%,Low,False,"Tokens [mutual, fund, sip] → 'investment_mutua..."
5,got interest from fd,NEFT,Credit,investment,fd_interest,investment,fd_interest,0.77,77.0%,High,True,"Tokens [got, interest, from] → 'investment_fd_..."
6,amazon pay txn,UPI,Debit,expense,shopping,expense,shopping,0.74,74.0%,High,True,"Tokens [amazon, pay, bit] → 'expense_shopping'..."
7,credited bonus amount,NEFT,Credit,income,salary,income,salary,0.78,78.0%,High,True,"Tokens [credited, bonus, amount] → 'income_sal..."
8,fuel payment hp petrol,UPI,Debit,expense,transport,expense,transport,0.82,82.0%,High,True,"Tokens [fuel, payment, petrol] → 'expense_tran..."
9,insurance premium paid,AUTO_DEBIT,Debit,expense,insurance,expense,insurance,0.68,68.0%,High,True,"Tokens [insurance, premium, paid] → 'expense_i..."


## | Metric | Meaning      | Use here               |
### | ------ | ------------ | ---------------------- |
### | F1     | balance      | good baseline          |
### | F2     | recall heavy | better for your system |


In [65]:
y_true = result_df["category"] + "_" + result_df["subcategory"]
y_pred = result_df["predicted_category"] + "_" + result_df["predicted_subcategory"]

In [66]:
from sklearn.metrics import f1_score, fbeta_score

f1 = f1_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Overall Metrics:")
print(f"F1 Score: {round(f1,3)}")
print(f"F2 Score: {round(f2,3)}")

Overall Metrics:
F1 Score: 0.942
F2 Score: 0.949


## Classification Report

In [67]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_true, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

print("\nClassification Report:")
print(report_df)


Classification Report:
                         precision    recall  f1-score     support
expense_bills             0.954545  1.000000  0.976744   21.000000
expense_food              1.000000  0.900000  0.947368   20.000000
expense_health            1.000000  1.000000  1.000000    7.000000
expense_insurance         1.000000  1.000000  1.000000   14.000000
expense_loan              0.869565  1.000000  0.930233   20.000000
expense_others            1.000000  0.500000  0.666667    2.000000
expense_shopping          0.904762  1.000000  0.950000   19.000000
expense_transport         1.000000  1.000000  1.000000   19.000000
income_others             1.000000  1.000000  1.000000    8.000000
income_salary             1.000000  1.000000  1.000000   24.000000
investment_fd_booking     0.900000  1.000000  0.947368    9.000000
investment_fd_interest    0.941176  1.000000  0.969697   16.000000
investment_mutual_funds   0.000000  0.000000  0.000000    4.000000
investment_others         0.000000  0.

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

## F2 Score per Class

In [68]:
from sklearn.metrics import fbeta_score

labels = list(set(y_true))

f2_scores = {}

for label in labels:
    y_true_bin = [1 if y == label else 0 for y in y_true]
    y_pred_bin = [1 if y == label else 0 for y in y_pred]

    f2_scores[label] = fbeta_score(y_true_bin, y_pred_bin, beta=2)

report_df["f2_score"] = report_df.index.map(f2_scores)

In [69]:
report_df = report_df.round(3)
print("\nClassification Report:")
print(report_df)


Classification Report:
                         precision  recall  f1-score  support  f2_score
expense_bills                0.955   1.000     0.977   21.000     0.991
expense_food                 1.000   0.900     0.947   20.000     0.918
expense_health               1.000   1.000     1.000    7.000     1.000
expense_insurance            1.000   1.000     1.000   14.000     1.000
expense_loan                 0.870   1.000     0.930   20.000     0.971
expense_others               1.000   0.500     0.667    2.000     0.556
expense_shopping             0.905   1.000     0.950   19.000     0.979
expense_transport            1.000   1.000     1.000   19.000     1.000
income_others                1.000   1.000     1.000    8.000     1.000
income_salary                1.000   1.000     1.000   24.000     1.000
investment_fd_booking        0.900   1.000     0.947    9.000     0.978
investment_fd_interest       0.941   1.000     0.970   16.000     0.988
investment_mutual_funds      0.000   0.0

## Best & Worst Performing Categories

In [70]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_health 1.0

Worst Performing Category:
investment_mutual_funds 0.0
